In [2]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer, EsmForTokenClassification, TrainingArguments, Trainer

# --- 1. Configuration and Model Initialization ---
# Switching to the 35M parameter version (12 layers, 480 hidden dim)
MODEL_CHECKPOINT = "facebook/esm2_t12_35M_UR50D"
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load model for binary token classification
model = EsmForTokenClassification.from_pretrained(MODEL_CHECKPOINT, num_labels=2).to(device)

disprot_database = pd.read_csv('F:/Disprot/Deepdisprot_train_combined_dataset.csv', keep_default_na=False)


def prepare_full_production_data(dataframe, max_seq_length=2000):
    """
    Cleans the dataset and extracts sequences/labels.
    Target: 3432 samples.
    """
    # 1. Remove rows where critical data is missing (NaN)
    valid_df = dataframe.dropna(subset=['Full_Sequence', 'Consensus']).copy()

    # Map characters to binary labels: 0 for Ordered (-), 1 for Disordered/Transition (D/T)
    label_dict = {'-': 0, 'T': 1, 'D': 1}
    production_data = {'seq': [], 'y': [], 'kw': []}
    
    for _, row in valid_df.iterrows():
        # Strip potential hidden whitespaces or line breaks
        seq = str(row['Full_Sequence']).strip()
        con = str(row['Consensus']).strip()
        
        # 2. Filter sequences that are too long (to avoid GPU memory crash)
        if len(seq) >= max_seq_length:
            continue
            
        # 3. Handle Excel Corruption Error (#NAME?)
        # If the label has been permanently corrupted to the string "#NAME?", 
        # the residue-level training data is lost and must be skipped.
        if con == "#NAME?":
            print(f"Warning: Label corrupted for {row['UniProt ACC']}. Please re-export the source file.")
            continue

        # 4. Length Consistency Check
        # If the file was read correctly, length should match, reaching the 3432 target.
        if len(seq) == len(con):
            production_data['seq'].append(seq)
            production_data['y'].append([label_dict.get(l, 0) for l in con])
            production_data['kw'].append(str(row.get('UniProt ACC', 'unknown')))
            
    print(f"Full Production Mode: Successfully prepared {len(production_data['seq'])} samples.")
    return production_data

Loading weights:   0%|          | 0/209 [00:00<?, ?it/s]

EsmForTokenClassification LOAD REPORT from: facebook/esm2_t12_35M_UR50D
Key                         | Status     | 
----------------------------+------------+-
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
classifier.bias             | MISSING    | 
classifier.weight           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [3]:
disprot_database.shape

(3432, 4)

In [4]:
full_train_data = prepare_full_production_data(disprot_database)

Full Production Mode: Successfully prepared 3376 samples.


In [5]:
def inspect_removed_records(dataframe, max_seq_length=2000):
    """
    Identifies and returns records that were filtered out during preparation.
    """
    # 1. Capture rows removed by NaN check
    initial_count = len(dataframe)
    nan_rows = dataframe[dataframe[['Full_Sequence', 'Consensus']].isna().any(axis=1)]
    valid_df = dataframe.dropna(subset=['Full_Sequence', 'Consensus']).copy()

    removed_data = [] # List to store why each record was removed
    
    for _, row in valid_df.iterrows():
        seq = str(row['Full_Sequence']).strip()
        con = str(row['Consensus']).strip()
        acc = str(row.get('UniProt ACC', 'unknown'))
        
        # 2. Check for Sequence Length
        if len(seq) >= max_seq_length:
            removed_data.append({'UniProt ACC': acc, 'Reason': 'Too Long', 'Length': len(seq)})
            continue
            
        # 3. Check for Excel Corruption
        if con == "#NAME?":
            removed_data.append({'UniProt ACC': acc, 'Reason': 'Excel #NAME? Corruption'})
            continue

        # 4. Check for Length Mismatch
        if len(seq) != len(con):
            removed_data.append({
                'UniProt ACC': acc, 
                'Reason': 'Length Mismatch', 
                'Seq_Len': len(seq), 
                'Con_Len': len(con)
            })
            
    # Convert to DataFrame for easy viewing
    removed_df = pd.DataFrame(removed_data)
    
    print(f"Total initial records: {initial_count}")
    print(f"Total records removed: {len(removed_df) + len(nan_rows)}")
    
    if len(nan_rows) > 0:
        print(f"- Missing Data (NaN): {len(nan_rows)} records")
    
    return removed_df

# Execute inspection
removed_records_summary = inspect_removed_records(disprot_database)

# Display the first 20 removed records to see the reasons
print("\n--- Top 20 Removed Records ---")
print(removed_records_summary.head(20))

# Optional: Save them to a CSV to inspect manually
# removed_records_summary.to_csv('removed_from_training.csv', index=False)


Total initial records: 3432
Total records removed: 56

--- Top 20 Removed Records ---
   UniProt ACC           Reason  Seq_Len  Con_Len
0     P61244-2  Length Mismatch      160      151
1     Q9Y237-2  Length Mismatch      131      156
2     Q8WXS3-1  Length Mismatch      145      180
3     P56211-2  Length Mismatch      112       96
4     P31431-2  Length Mismatch      198      153
5     O54918-2  Length Mismatch      196      140
6     Q9HB71-2  Length Mismatch      228       80
7     P46108-2  Length Mismatch      304      204
8     O00204-2  Length Mismatch      365      350
9     P04370-5  Length Mismatch      250      169
10    Q9NS23-4  Length Mismatch      344      270
11    P02686-5  Length Mismatch      304      171
12    P61328-2  Length Mismatch      243      181
13    P33316-2  Length Mismatch      252      164
14    P63277-2  Length Mismatch      210       75
15    Q96PU8-9  Length Mismatch      341      319
16    Q92784-2  Length Mismatch      378      357
17    Q92913-2

In [6]:
# --- 3. Dataset Definition ---
class ESM2DisorderDataset(Dataset):
    def __init__(self, data_split, tokenizer, max_len=2000):
        self.sequences = data_split['seq']
        self.labels = data_split['y']
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self): return len(self.sequences)

    def __getitem__(self, idx):
        seq, label = self.sequences[idx], self.labels[idx]
        encoding = self.tokenizer(seq, truncation=True, max_length=self.max_len, padding="max_length", return_tensors="pt")
        padded_labels = torch.full((self.max_len,), -100, dtype=torch.long)
        seq_len = min(len(label), self.max_len - 2)
        padded_labels[1 : seq_len + 1] = torch.tensor(label[:seq_len], dtype=torch.long)
        return {'input_ids': encoding['input_ids'].squeeze(), 'attention_mask': encoding['attention_mask'].squeeze(), 'labels': padded_labels}

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from torch.utils.data import Dataset
from transformers import AutoTokenizer, EsmForTokenClassification, TrainingArguments, Trainer
from tqdm import tqdm
# --- 2. Advanced Focal + TV Loss Implementation ---
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean', smooth_weight=0.1):
        super(FocalLoss, self).__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.reduction = reduction
        self.smooth_weight = smooth_weight
    
    def forward(self, inputs, targets):
        # 1. Focal Loss
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = (1 - pt)**self.gamma * ce_loss
        
        if self.alpha is not None:
            if self.alpha.device != targets.device:
                self.alpha = self.alpha.to(targets.device)
            alpha_t = self.alpha.gather(0, targets.long())
            focal_loss = alpha_t * focal_loss
        
        # 2. Total Variation (TV) Loss for Smoothness
        tv_loss = 0
        if self.smooth_weight > 0 and inputs.size(0) > 1:
            probs = F.softmax(inputs, dim=-1)
            # Penalize difference between adjacent residues
            diff = torch.abs(probs[1:] - probs[:-1])
            tv_loss = torch.mean(diff)
            
        if self.reduction == 'mean':
            return torch.mean(focal_loss) + self.smooth_weight * tv_loss
        return torch.sum(focal_loss) + (self.smooth_weight * tv_loss * inputs.size(0))

In [8]:
import torch

def get_imbalance_weights(data_all):
    """
    Counts total 0s and 1s across all sequences and calculates alpha.
    """
    # Flatten all labels into one single tensor
    # We use data_all['y'] which is a list of tensors
    all_labels = torch.cat([torch.tensor(y) if not isinstance(y, torch.Tensor) else y for y in data_all['y']])
    
    # Count occurrences
    count_0 = (all_labels == 0).sum().item()
    count_1 = (all_labels == 1).sum().item()
    total = count_0 + count_1
    
    # Calculate inverse frequency weights
    # Formula: total_samples / (num_classes * class_samples)
    alpha_0 = total / (2.0 * count_0)
    alpha_1 = total / (2.0 * count_1)
    
    alpha = torch.tensor([alpha_0, alpha_1], dtype=torch.float)
    
    print(f"--- Class Imbalance Report ---")
    print(f"Class 0 (Ordered): {count_0} ({count_0/total:.2%})")
    print(f"Class 1 (Disordered): {count_1} ({count_1/total:.2%})")
    print(f"Calculated alpha: {alpha}")
    
    return alpha

# Execute using your prepared 3376 samples
alpha_values = get_imbalance_weights(full_train_data)


--- Class Imbalance Report ---
Class 0 (Ordered): 1350860 (80.17%)
Class 1 (Disordered): 334136 (19.83%)
Calculated alpha: tensor([0.6237, 2.5214])


In [9]:
# --- 3. Custom Trainer for Sequential TV Loss ---
class DisorderTrainer(Trainer):
    def __init__(self, alpha=None, gamma=2.0, smooth_weight=0.1, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.custom_loss_fct = FocalLoss(alpha=alpha, gamma=gamma, smooth_weight=smooth_weight)

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        loss = 0
        valid_count = 0
        # Calculate loss per sequence to maintain TV Loss continuity
        for i in range(logits.shape[0]):
            mask = labels[i] != -100
            if not mask.any(): continue
            
            active_logits = logits[i][mask]
            active_labels = labels[i][mask]
            
            loss += self.custom_loss_fct(active_logits, active_labels)
            valid_count += 1

        final_loss = loss / valid_count if valid_count > 0 else loss
        return (final_loss, outputs) if return_outputs else final_loss

# --- 4. Data Preparation & Alpha Calculation ---
def get_alpha_weights(data):
    all_labels = torch.cat([torch.tensor(y) for y in data['y']])
    c0, c1 = (all_labels == 0).sum().item(), (all_labels == 1).sum().item()
    total = c0 + c1
    alpha = torch.tensor([total/(2.0*c0), total/(2.0*c1)], dtype=torch.float)
    print(f"Alpha Weights: Ordered={alpha[0]:.2f}, Disordered={alpha[1]:.2f}")
    return alpha

# Load and prepare full_train_data (assumes disprot_database is loaded)
# full_train_data = prepare_full_production_data(disprot_database) 
alpha_values = get_alpha_weights(full_train_data)

Alpha Weights: Ordered=0.62, Disordered=2.52


In [ ]:
# --- 5. Execution ---
training_args = TrainingArguments(
    output_dir="./esm2_35M_focal_tv_production",
    num_train_epochs=3,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    fp16=torch.cuda.is_available(),
    logging_steps=10,
    save_strategy="epoch",
    report_to="none"
)

# Build Dataset
train_dataset = ESM2DisorderDataset(full_train_data, tokenizer)

# Initialize Custom Trainer
trainer = DisorderTrainer(
    alpha=alpha_values,
    gamma=2.0,
    smooth_weight=0.1, # TV Loss strength
    model=model,
    args=training_args,
    train_dataset=train_dataset,
)

print("Starting Advanced Training (Focal + TV Loss)...")
trainer.train()
trainer.save_model("./esm2_35M_disorder_focal_tv_final")

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix

def evaluate_disorder_model(data_split, model, tokenizer, device):
    """
    Predicts the consensus for the entire dataset and generates evaluation metrics.
    """
    model.eval()
    all_predictions = []
    all_true_labels = []
    
    print(f"Starting prediction for {len(data_split['seq'])} sequences...")
    
    with torch.no_grad():
        for i, seq in enumerate(tqdm(data_split['seq'])):
            # 1. Get ground truth labels
            true_y = data_split['y'][i]
            
            # 2. Tokenize and run inference
            inputs = tokenizer(seq, return_tensors="pt", truncation=True, max_length=2000).to(device)
            outputs = model(**inputs)
            logits = outputs.logits  # Shape: [1, L+2, 2]
            
            # 3. Get predictions and remove <cls> and <eos> tokens
            # We take the argmax across the 2 labels (0 or 1)
            preds = torch.argmax(logits, dim=-1).squeeze(0)
            # Slice [1:-1] to match the original sequence length
            preds_cleaned = preds[1:-1].cpu().numpy().tolist()
            
            # Ensure lengths match (handling rare truncation edge cases)
            valid_len = min(len(preds_cleaned), len(true_y))
            all_predictions.extend(preds_cleaned[:valid_len])
            all_true_labels.extend(true_y[:valid_len])

    # --- 1. Classification Summary Report ---
    print("\n" + "="*60)
    print("RESIDUE-LEVEL CLASSIFICATION REPORT")
    print("="*60)
    target_names = ['Ordered (0)', 'Disordered (1)']
    print(classification_report(all_true_labels, all_predictions, target_names=target_names))

    # --- 2. Confusion Matrix Visualization ---
    cm = confusion_matrix(all_true_labels, all_predictions)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', 
                xticklabels=target_names, yticklabels=target_names)
    
    plt.title('Confusion Matrix - Full Dataset (Focal + TV Loss)', fontsize=14, fontweight='bold')
    plt.ylabel('Actual Label', fontsize=12)
    plt.xlabel('Predicted Label', fontsize=12)
    plt.tight_layout()
    plt.show()
    
    return all_true_labels, all_predictions

# Run the evaluation
y_true_final, y_pred_final = evaluate_disorder_model(full_train_data, model, tokenizer, device)


Starting prediction for 3376 sequences...


 50%|████▉     | 1673/3376 [02:02<03:52,  7.32it/s]

In [ ]:
external_database = pd.read_csv('Disprot/Deepdisprot_external_combined_dataset.csv', keep_default_na=False)

In [ ]:
def prepare_test_set_unconstrained(dataframe):
    valid_df = dataframe.dropna(subset=['Full_Sequence', 'Consensus']).copy()
    label_dict = {'-': 0, 'T': 1, 'D': 1}
    test_data = {'seq': [], 'y': [], 'kw': []}
    
    for _, row in valid_df.iterrows():
        seq = str(row['Full_Sequence']).strip()
        con = str(row['Consensus']).strip()
        

        if con == "#NAME?" or len(seq) != len(con):
            continue
            
        test_data['seq'].append(seq)
        test_data['y'].append([label_dict.get(l, 0) for l in con])
        test_data['kw'].append(str(row.get('UniProt ACC', 'unknown')))
            
    print(f"Test Set Ready: {len(test_data['seq'])} samples.")
    return test_data

external_test_data = prepare_test_set_unconstrained(external_database)

In [ ]:
y_true_ext, y_pred_ext = evaluate_disorder_model(external_test_data, model, tokenizer, device)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import random

def plot_disorder_comparison_heatmaps(data_split, model, tokenizer, device, num_samples=10):
    model.eval()
    indices = random.sample(range(len(data_split['seq'])), num_samples)
    fig, axes = plt.subplots(num_samples, 1, figsize=(15, 2 * num_samples))
    if num_samples == 1: axes = [axes] 
    
    with torch.no_grad():
        for i, idx in enumerate(indices):
            seq = data_split['seq'][idx]
            true_y = np.array(data_split['y'][idx]).reshape(1, -1)
            protein_id = data_split['kw'][idx]
            inputs = tokenizer(seq, return_tensors="pt", truncation=True, max_length=2000).to(device)
            logits = model(**inputs).logits
            preds = torch.argmax(logits, dim=-1).squeeze(0)[1:-1].cpu().numpy().reshape(1, -1)
            min_len = min(true_y.shape[1], preds.shape[1])
            plot_data = np.vstack([true_y[:, :min_len], preds[:, :min_len]])
            sns.heatmap(plot_data, cmap="YlGnBu", cbar=False, ax=axes[i], 
                        xticklabels=False, yticklabels=['True', 'Pred'])
            
            axes[i].set_title(f"Protein: {protein_id} (Length: {min_len})", fontsize=10, loc='left')
            
    plt.tight_layout()
    plt.show()

plot_disorder_comparison_heatmaps(external_test_data, model, tokenizer, device, num_samples=10)


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

def plot_top_10_matching_proteins(data_split, model, tokenizer, device):
    model.eval()
    results = []

    print("正在計算每條蛋白質的預測準確度...")
    with torch.no_grad():
        for i in tqdm(range(len(data_split['seq']))):
            seq = data_split['seq'][i]
            true_y = data_split['y'][i]
            protein_id = data_split['kw'][i]
            
            # 模型推理
            # 這裡暫時維持 2000 限制以防 GPU 崩潰，若需更長請改用 CPU
            inputs = tokenizer(seq, return_tensors="pt", truncation=True, max_length=2000).to(device)
            logits = model(**inputs).logits
            preds = torch.argmax(logits, dim=-1).squeeze(0)[1:-1].cpu().numpy()
            
            # 計算該蛋白質的一致性得分 (Accuracy)
            true_y_arr = np.array(true_y[:len(preds)])
            if len(true_y_arr) == 0: continue
            
            score = np.mean(true_y_arr == preds)
            results.append({
                'id': protein_id,
                'score': score,
                'true': true_y_arr,
                'pred': preds,
                'len': len(true_y_arr)
            })

    # 根據分數從高到低排序，取前 10 名
    top_10 = sorted(results, key=lambda x: x['score'], reverse=True)[:10]

    # --- 繪製熱圖 ---
    fig, axes = plt.subplots(10, 1, figsize=(15, 18))
    
    for i, res in enumerate(top_10):
        # 準備繪圖數據 [2, L]
        plot_data = np.vstack([res['true'].reshape(1, -1), res['pred'].reshape(1, -1)])
        
        sns.heatmap(plot_data, cmap="YlGnBu", cbar=False, ax=axes[i], 
                    xticklabels=False, yticklabels=['True', 'Pred'])
        
        axes[i].set_title(f"Rank {i+1}: {res['id']} | Accuracy: {res['score']:.2%} | Length: {res['len']}", 
                          fontsize=10, loc='left', fontweight='bold')

    plt.suptitle("Top 10 Best Matching Proteins (Sorted by Accuracy)", fontsize=16, y=1.02)
    plt.tight_layout()
    plt.show()

# 執行分析
plot_top_10_matching_proteins(external_test_data, model, tokenizer, device)


In [ ]:
from sklearn.metrics import f1_score

def plot_top_10_complex_proteins(data_split, model, tokenizer, device):
    model.eval()
    complex_results = []

    print("Searching for complex proteins with high prediction overlap...")
    with torch.no_grad():
        for i in tqdm(range(len(data_split['seq']))):
            seq = data_split['seq'][i]
            true_y = np.array(data_split['y'][i])
            protein_id = data_split['kw'][i]
            
            # 1. FILTER: Must contain BOTH 0 and 1 in ground truth
            if len(np.unique(true_y)) < 2:
                continue

            inputs = tokenizer(seq, return_tensors="pt", truncation=True, max_length=2000).to(device)
            logits = model(**inputs).logits
            preds = torch.argmax(logits, dim=-1).squeeze(0)[1:-1].cpu().numpy()
            
            # Align lengths
            min_len = min(len(preds), len(true_y))
            y_t = true_y[:min_len]
            y_p = preds[:min_len]
            
            # 2. RANK: Use F1-score specifically for the Disordered class (1)
            # This ensures the model actually caught the blue blocks correctly
            score = f1_score(y_t, y_p, zero_division=0)
            
            complex_results.append({
                'id': protein_id,
                'score': score,
                'true': y_t,
                'pred': y_p,
                'len': min_len
            })

    # Sort by F1-score (highest overlap of disorder regions)
    top_10_complex = sorted(complex_results, key=lambda x: x['score'], reverse=True)[:10]

    # --- Plotting ---
    fig, axes = plt.subplots(10, 1, figsize=(15, 18))
    for i, res in enumerate(top_10_complex):
        plot_data = np.vstack([res['true'].reshape(1, -1), res['pred'].reshape(1, -1)])
        sns.heatmap(plot_data, cmap="YlGnBu", cbar=False, ax=axes[i], 
                    xticklabels=False, yticklabels=['True', 'Pred'])
        
        axes[i].set_title(f"Rank {i+1}: {res['id']} | F1-Score: {res['score']:.2%} | Length: {res['len']}", 
                          fontsize=10, loc='left', fontweight='bold')

    plt.suptitle("Top 10 Complex Proteins (Ordered & Disordered Mix)", fontsize=16, y=1.02)
    plt.tight_layout()
    plt.show()

# Run this to see the real "mixed" success stories
plot_top_10_complex_proteins(external_test_data, model, tokenizer, device)


In [ ]:
def evaluate_and_extract_embeddings(data_split, model, tokenizer, device):
    """
    Predicts labels AND extracts ESM-2 embeddings for the dataset.
    Returns: y_true, y_pred, and a dictionary of embeddings {protein_id: np.array}.
    """
    model.eval()
    all_predictions = []
    all_true_labels = []
    embeddings_dict = {} # To store the vectors
    
    print(f"Inference & Embedding extraction for {len(data_split['seq'])} sequences...")
    
    with torch.no_grad():
        for i, seq in enumerate(tqdm(data_split['seq'])):
            true_y = data_split['y'][i]
            protein_id = data_split['kw'][i]
            
            # 1. Forward pass with hidden states enabled
            inputs = tokenizer(seq, return_tensors="pt", truncation=True, max_length=2000).to(device)
            outputs = model(**inputs, output_hidden_states=True)
            
            # 2. Extract Predictions (from logits)
            logits = outputs.logits
            preds = torch.argmax(logits, dim=-1).squeeze(0)[1:-1].cpu().numpy().tolist()
            
            # 3. Extract Embeddings (from the last hidden layer)
            # Shape: [1, L+2, Hidden_Dim] -> we remove <cls> and <eos>
            # Resulting shape: [L, Hidden_Dim] (e.g., [L, 480] for 35M model)
            last_hidden_state = outputs.hidden_states[-1].squeeze(0)[1:-1, :].cpu().numpy()
            embeddings_dict[protein_id] = last_hidden_state

            # Alignment for report
            valid_len = min(len(preds), len(true_y))
            all_predictions.extend(preds[:valid_len])
            all_true_labels.extend(true_y[:valid_len])

    # --- Print Report ---
    print("\n" + "="*60)
    print("RESIDUE-LEVEL CLASSIFICATION REPORT")
    print("="*60)
    print(classification_report(all_true_labels, all_predictions, target_names=['Ordered', 'Disordered']))

    return all_true_labels, all_predictions, embeddings_dict

# Run the dual process
y_true, y_pred, protein_embeddings = evaluate_and_extract_embeddings(full_train_data, model, tokenizer, device)

# Save embeddings to disk
import numpy as np
np.savez_compressed("trained_esm2_embeddings.npz", **protein_embeddings)
print("Embeddings saved to trained_esm2_embeddings.npz")


In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

def prepare_rf_data(embeddings_dict, data_split):
    """
    Flattens the dictionary of [L, Hidden_Dim] embeddings into 
    a single X (features) matrix and y (labels) vector.
    """
    X_list = []
    y_list = []
    
    print("Flattening embeddings for Random Forest...")
    for i, kw in enumerate(data_split['kw']):
        if kw in embeddings_dict:
            # Get the embedding [L, Hidden_Dim]
            emb = embeddings_dict[kw]
            # Get the labels [L]
            labels = np.array(data_split['y'][i])
            
            # Align lengths in case of truncation
            min_len = min(len(emb), len(labels))
            X_list.append(emb[:min_len])
            y_list.append(labels[:min_len])
            
    # Combine all residues into one giant matrix
    X = np.vstack(X_list)
    y = np.concatenate(y_list)
    
    print(f"Data ready. Total residues: {X.shape[0]}, Features: {X.shape[1]}")
    return X, y

# Using the 'protein_embeddings' extracted in the previous step
#X_train, y_train = prepare_rf_data(protein_embeddings, full_train_data)


In [ ]:
# Initialize Random Forest
rf_model = RandomForestClassifier(
    n_estimators=100,      # Number of trees
    max_depth=15,          # Limit depth to prevent overfitting and save memory
    n_jobs=-1,             # Use all CPU cores for speed
    random_state=42,
    class_weight='balanced', # Automatically adjusts weights for Order/Disorder imbalance
    verbose=1
)

print("Training Random Forest on ESM-2 embeddings...")
rf_model.fit(X_train, y_train)

# Quick check on Training Set
y_train_pred = rf_model.predict(X_train)
print("\n--- Random Forest Training Performance ---")
print(classification_report(y_train, y_train_pred, target_names=['Ordered', 'Disordered']))


In [ ]:
_, _, ext_embeddings_dict = evaluate_and_extract_embeddings(external_test_data, model, tokenizer, device)
X_test_ext, y_test_ext = prepare_rf_data(ext_embeddings_dict, external_test_data)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

y_test_pred_rf = rf_model.predict(X_test_ext)


print("\n" + "="*60)
print("RANDOM FOREST - EXTERNAL DATASET PERFORMANCE")
print("="*60)
print(classification_report(y_test_ext, y_test_pred_rf, target_names=['Ordered', 'Disordered']))


cm_rf = confusion_matrix(y_test_ext, y_test_pred_rf)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Oranges', 
            xticklabels=['Ordered', 'Disordered'], yticklabels=['Ordered', 'Disordered'])
plt.title('RF Confusion Matrix - External Dataset', fontsize=14, fontweight='bold')
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.show()

In [ ]:
# Save embeddings to disk
import numpy as np
np.savez_compressed("trained_esm2_embeddings_external.npz", **ext_embeddings_dict)
print("Embeddings saved to trained_esm2_embeddings_external.npz")

In [ ]:
from sklearn.metrics import f1_score
import torch

def plot_top_10_complex_proteins_RF(data_split, esm_model, rf_model, tokenizer, device):
    """
    Finds and plots top 10 complex proteins using Random Forest predictions.
    """
    esm_model.eval()
    complex_results = []

    print("Extracting features and predicting with Random Forest...")
    with torch.no_grad():
        for i in tqdm(range(len(data_split['seq']))):
            seq = data_split['seq'][i]
            true_y = np.array(data_split['y'][i])
            protein_id = data_split['kw'][i]
            
            # 1. FILTER: Ground truth must contain both Ordered and Disordered
            if len(np.unique(true_y)) < 2:
                continue

            # 2. FEATURE EXTRACTION: Get ESM-2 Embeddings
            inputs = tokenizer(seq, return_tensors="pt", truncation=True, max_length=2000).to(device)
            outputs = esm_model(**inputs, output_hidden_states=True)
            # Shape: [L, Hidden_Dim]
            embeddings = outputs.hidden_states[-1].squeeze(0)[1:-1, :].cpu().numpy()
            
            # 3. RF PREDICTION: Use the Random Forest model
            # RF expects [L, Hidden_Dim]
            preds = rf_model.predict(embeddings)
            
            # Align lengths
            min_len = min(len(preds), len(true_y))
            y_t = true_y[:min_len]
            y_p = preds[:min_len]
            
            # 4. RANK: Use F1-score for the Disordered class
            score = f1_score(y_t, y_p, zero_division=0)
            
            complex_results.append({
                'id': protein_id,
                'score': score,
                'true': y_t,
                'pred': y_p,
                'len': min_len
            })

    # Sort by F1-score
    top_10_complex = sorted(complex_results, key=lambda x: x['score'], reverse=True)[:10]

    # --- Plotting ---
    fig, axes = plt.subplots(10, 1, figsize=(15, 18))
    for i, res in enumerate(top_10_complex):
        plot_data = np.vstack([res['true'].reshape(1, -1), res['pred'].reshape(1, -1)])
        sns.heatmap(plot_data, cmap="YlGnBu", cbar=False, ax=axes[i], 
                    xticklabels=False, yticklabels=['True', 'Pred'])
        
        axes[i].set_title(f"Rank {i+1}: {res['id']} | RF F1-Score: {res['score']:.2%} | Length: {res['len']}", 
                          fontsize=10, loc='left', fontweight='bold')

    plt.suptitle("Top 10 Complex Proteins (Random Forest + ESM-2 Embeddings)", fontsize=16, y=1.02)
    plt.tight_layout()
    plt.show()

# Run the updated RF evaluation
# Pass both your fine-tuned ESM model and the trained RF model
plot_top_10_complex_proteins_RF(external_test_data, model, rf_model, tokenizer, device)

In [ ]:
import xgboost as xgb
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import time

def load_embeddings_from_npz(file_path):
    print(f"Loading embeddings from {file_path}...")
    data = np.load(file_path, allow_pickle=True)
    embeddings_dict = {key: data[key] for key in data.files}
    
    print(f"Successfully loaded {len(embeddings_dict)} protein embeddings.")
    return embeddings_dict


protein_embeddings = load_embeddings_from_npz("trained_esm2_embeddings.npz")
ext_embeddings_dict = load_embeddings_from_npz("trained_esm2_embeddings_external.npz")

X_train, y_train = prepare_rf_data(protein_embeddings, full_train_data)
X_test_ext, y_test_ext = prepare_rf_data(ext_embeddings_dict, external_test_data)


xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    tree_method='hist',    
    device='cuda',
    max_bin=256,
    scale_pos_weight=None,  
    random_state=42,
    verbosity=1
)

print("Training XGBoost on GPU with ESM-2 embeddings...")
start_time = time.time()

xgb_model.fit(X_train, y_train)

end_time = time.time()
duration_seconds = end_time - start_time
duration_minutes = duration_seconds / 60
throughput = X_train.shape[0] / duration_seconds

print("\n" + "="*30)
print("TRAINING COMPLETED")
print("="*30)
print(f"⏱️  Total Training Time: {duration_seconds:.2f} seconds ({duration_minutes:.2f} minutes)")
print(f"🚀 Speed: {throughput:.2f} residues/second")
print("="*30)
y_test_pred_xgb = xgb_model.predict(X_test_ext)

print("\n" + "="*60)
print("XGBOOST - EXTERNAL DATASET PERFORMANCE")
print("="*60)
print(classification_report(y_test_ext, y_test_pred_xgb, target_names=['Ordered', 'Disordered']))

cm_xgb = confusion_matrix(y_test_ext, y_test_pred_xgb)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_xgb, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Ordered', 'Disordered'], yticklabels=['Ordered', 'Disordered'])
plt.title('XGBoost Confusion Matrix (GPU) - External Dataset', fontsize=14, fontweight='bold')
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import f1_score

def plot_consensus_heatmaps(data_split, y_pred, n_examples=6):
    """
    Plots heatmaps comparing True vs Predicted consensus, 
    matching the style of your provided example.
    """
    current_idx = 0
    results = []
    
    # 1. Reconstruct and Calculate Metrics
    for i, kw in enumerate(data_split['kw']):
        true_labels = np.array(data_split['y'][i])
        length = len(true_labels)
        prot_pred = y_pred[current_idx : current_idx + length]
        current_idx += length
        
        # Alignment check
        min_len = min(len(true_labels), len(prot_pred))
        y_true_adj = true_labels[:min_len]
        y_pred_adj = prot_pred[:min_len]
        
        if min_len == 0: continue
        
        # Only keep examples with both classes for a meaningful "consensus" plot
        if len(np.unique(y_true_adj)) > 1:
            score = f1_score(y_true_adj, y_pred_adj, average='binary')
            results.append({
                'id': kw,
                'true': y_true_adj,
                'pred': y_pred_adj,
                'f1': score,
                'len': min_len
            })

    # 2. Sort by F1 Score (Highest first)
    results = sorted(results, key=lambda x: x['f1'], reverse=True)

    # 3. Plotting
    n_to_plot = min(n_examples, len(results))
    fig, axes = plt.subplots(n_to_plot, 1, figsize=(12, 2 * n_to_plot))
    if n_to_plot == 1: axes = [axes]

    for idx, res in enumerate(results[:n_to_plot]):
        # Create a 2D array for the heatmap: [Row 0 = True, Row 1 = Pred]
        heatmap_data = np.vstack([res['true'], res['pred']])
        
        # Use 'YlGnBu' or 'cividis' for that dark blue/yellow look
        im = axes[idx].imshow(heatmap_data, aspect='auto', cmap='YlGnBu', interpolation='nearest')
        
        axes[idx].set_title(f"Rank {idx+1}: {res['id']} | F1-Score: {res['f1']:.2%} | Length: {res['len']}", 
                            fontsize=10, fontweight='bold', loc='left')
        
        # Formatting labels
        axes[idx].set_yticks([0, 1])
        axes[idx].set_yticklabels(['True', 'Pred'], fontsize=8)
        axes[idx].set_xticks([]) # Hide sequence index for cleaner look
        
    plt.tight_layout()
    plt.show()

# Execute the visualization
plot_consensus_heatmaps(external_test_data, y_test_pred_xgb, n_examples=6)

